In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prathamtripathi/drug-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'drug-classification' dataset.
Path to dataset files: /kaggle/input/drug-classification


In [2]:
import pandas as pd
import os

# Assuming the dataset contains a CSV file, let's find it.
# The 'path' variable from the previous cell should point to a directory.
# We can list the contents of that directory to find the CSV file.

dataset_files = os.listdir(path)
csv_file = None
for file in dataset_files:
    if file.endswith('.csv'):
        csv_file = os.path.join(path, file)
        break

if csv_file:
    df = pd.read_csv(csv_file)
    print(f"Successfully loaded data from: {csv_file}")
    display(df.head())
else:
    print("No CSV file found in the downloaded dataset.")

Successfully loaded data from: /kaggle/input/drug-classification/drug200.csv


,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,DrugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,DrugY


In [3]:
# skewness
num_cols = df.select_dtypes(include = ['int64' , 'float64']).columns.tolist()
skewed_df = df[num_cols].skew()
print(skewed_df)

Age        0.030308
Na_to_K    1.039341
dtype: float64


In [4]:
# 1. Numerical cols we have to apply skewness transformation (-> Apply Skewness Tranformation / StandardScaler)
skewed_num_feature = skewed_df[skewed_df > 0.5].index.tolist()
print(skewed_num_feature)

# 2. Columns that is not going to be processed for skewness Transformation (-> Only we use StandardScaler )
other_num_feature = skewed_df[skewed_df <= 0.5 ].index.tolist()
print(other_num_feature)

# 3. Object Cols but without 'Drug' Because that is our target colums (-> Have to One Hot encoding)
cat_feature = df.select_dtypes(include = ['object']).drop('Drug', axis = 1).columns.tolist()
print(cat_feature)

['Na_to_K']
['Age']
['Sex', 'BP', 'Cholesterol']


In [5]:
# Building

# Building Pipelines for our 3 Columns

1. Skewed features (1. skewness Transformation ,

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [7]:
# 1. Pipeline for skewed Transformer & standard Scaler -> skewed_num_feature
skewed_transformer = Pipeline(steps = [
    ('power_Transformer', PowerTransformer()),
    ('standard_scaler' , StandardScaler())

])

# 2. Pipeline for other num feature
other_num_transformer = Pipeline(steps = [
    ('standard_scaler', StandardScaler())
])

# 3. pipeline for cat feature
cat_transformer = Pipeline(steps = [
    ('one_hot_encoder', OneHotEncoder())
])

preprocessor = ColumnTransformer(transformers = [
    ('skewed_num', skewed_transformer, skewed_num_feature),
    ('other_num' , other_num_transformer, other_num_feature),
    ('cat_feature' , cat_transformer, cat_feature)]
    , remainder='passthrough')

skewed_transformer

Pipeline(steps=[('power_Transformer', PowerTransformer()),
                ('standard_scaler', StandardScaler())])

In [8]:
other_num_transformer

Pipeline(steps=[('standard_scaler', StandardScaler())])

In [9]:
cat_transformer

Pipeline(steps=[('one_hot_encoder', OneHotEncoder())])

In [10]:
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('skewed_num',
                                 Pipeline(steps=[('power_Transformer',
                                                  PowerTransformer()),
                                                 ('standard_scaler',
                                                  StandardScaler())]),
                                 ['Na_to_K']),
                                ('other_num',
                                 Pipeline(steps=[('standard_scaler',
                                                  StandardScaler())]),
                                 ['Age']),
                                ('cat_feature',
                                 Pipeline(steps=[('one_hot_encoder',
                                                  OneHotEncoder())]),
                                 ['Sex', 'BP', 'Cholesterol'])])

In [12]:
X = df.drop('Drug', axis = 1)
y = df['Drug']

In [13]:
X_Transformed = preprocessor.fit_transform(X)
print(X_Transformed)

[[ 1.27076331 -1.29159102  1.         ...  0.          1.
   0.        ]
 [-0.20895238  0.16269866  0.         ...  0.          1.
   0.        ]
 [-0.8851916   0.16269866  0.         ...  0.          1.
   0.        ]
 ...
 [-0.94539752  0.46567567  0.         ...  1.          1.
   0.        ]
 [-0.03918172 -1.29159102  0.         ...  1.          0.
   1.        ]
 [-0.57645012 -0.26146916  1.         ...  0.          0.
   1.        ]]


In [14]:
import plotly.express as px
from sklearn.decomposition import PCA

# Get feature names after transformation
transformed_feature_names = preprocessor.get_feature_names_out()

# Create a DataFrame for the transformed features
X_transformed_df = pd.DataFrame(X_Transformed, columns=transformed_feature_names)

# Add the target variable 'y' to the transformed DataFrame for coloring
X_transformed_df['Drug'] = y.values

display(X_transformed_df.head())

,skewed_num__Na_to_K,other_num__Age,cat_feature__Sex_F,cat_feature__Sex_M,cat_feature__BP_HIGH,cat_feature__BP_LOW,cat_feature__BP_NORMAL,cat_feature__Cholesterol_HIGH,cat_feature__Cholesterol_NORMAL,Drug
0,1.270763,-1.291591,1.0,0.0,1.0,0.0,0.0,1.0,0.0,DrugY
1,-0.208952,0.162699,0.0,1.0,0.0,1.0,0.0,1.0,0.0,drugC
2,-0.885192,0.162699,0.0,1.0,0.0,1.0,0.0,1.0,0.0,drugC
3,-1.622866,-0.988614,1.0,0.0,0.0,0.0,1.0,1.0,0.0,drugX
4,0.553205,1.011034,1.0,0.0,0.0,1.0,0.0,1.0,0.0,DrugY


### 2D Visualization of Transformed Data

Let's visualize two of the transformed numerical features ('Na_to_K' and 'Age') to see how the different drug types are distributed after preprocessing.

In [22]:
# Identify the transformed 'Na_to_K' and 'Age' columns
# The names will include the transformer prefix, e.g., 'skewed_num__Na_to_K' and 'other_num__Age'

na_to_k_col = [col for col in transformed_feature_names if 'Na_to_K' in col][0]
age_col = [col for col in transformed_feature_names if 'Age' in col][0]

fig_2d = px.scatter(
    X_transformed_df,
    x=na_to_k_col,
    y=age_col,
    color='Drug',
    title='2D Scatter Plot of Transformed Na_to_K vs. Age',
    labels={na_to_k_col: 'Transformed Na_to_K', age_col: 'Transformed Age'},
    hover_data={'Drug': True, na_to_k_col: ':.2f', age_col: ':.2f'},
    width=800,
    height=500,
    template='plotly_dark'
)
fig_2d.show()

### 3D Visualization of Transformed Data (using PCA)

To make it even more insightful and "cool", we'll use Principal Component Analysis (PCA) to reduce the dimensionality of the fully transformed data to 3 components. This allows us to visualize the high-dimensional data in 3D, showing how the drug classes separate in the most significant directions of variance.

In [23]:
# Apply PCA to reduce dimensions for 3D visualization
pca = PCA(n_components=3)
X_pca_3d = pca.fit_transform(X_Transformed)

# Create a DataFrame for PCA results
pca_df = pd.DataFrame(
    X_pca_3d,
    columns=['Principal Component 1', 'Principal Component 2', 'Principal Component 3']
)
pca_df['Drug'] = y.values

fig_3d = px.scatter_3d(
    pca_df,
    x='Principal Component 1',
    y='Principal Component 2',
    z='Principal Component 3',
    color='Drug',
    title='3D PCA of Transformed Drug Data',
    labels={
        'Principal Component 1': 'PC1',
        'Principal Component 2': 'PC2',
        'Principal Component 3': 'PC3'
    },
    width=900,
    height=700,
    template='plotly_dark'
)
fig_3d.show()

### Visualizing Pipeline Effects: 2D Transformation Journey for 'Na_to_K'

Let's visualize how the `Na_to_K` feature changes its distribution as it passes through the `skewed_transformer` pipeline (PowerTransformer followed by StandardScaler).

In [24]:
import numpy as np

# Extract the original 'Na_to_K' values
na_to_k_original_values_df = X[['Na_to_K']]

# Get the PowerTransformer and StandardScaler from the preprocessor
power_transformer = preprocessor.named_transformers_['skewed_num'].named_steps['power_Transformer']
standard_scaler = preprocessor.named_transformers_['skewed_num'].named_steps['standard_scaler']

# Apply PowerTransformer (passing DataFrame for cleaner handling of feature names)
na_to_k_power_transformed_values = power_transformer.transform(na_to_k_original_values_df)

# Apply StandardScaler to the power-transformed values
na_to_k_standard_scaled_values = standard_scaler.transform(na_to_k_power_transformed_values)

# Create a DataFrame to hold all stages for plotting
na_to_k_stages_df = pd.DataFrame({
    'Value': np.concatenate([
        na_to_k_original_values_df['Na_to_K'].values,
        na_to_k_power_transformed_values.flatten(),
        na_to_k_standard_scaled_values.flatten()
    ]),
    'Stage': np.concatenate([
        np.full(len(na_to_k_original_values_df), 'Original'),
        np.full(len(na_to_k_power_transformed_values), 'Power Transformed'),
        np.full(len(na_to_k_standard_scaled_values), 'Standard Scaled')
    ]),
    'Drug': np.concatenate([y, y, y]) # Include 'Drug' for consistent coloring
})

# Create the 2D violin plot
fig_na_to_k_journey = px.violin(
    na_to_k_stages_df,
    x='Stage',
    y='Value',
    color='Stage',
    box=True, # show box plots inside violins
    points='outliers', # show all points for outliers
    title='Distribution of Na_to_K at Different Pipeline Stages',
    labels={'Value': 'Feature Value', 'Stage': 'Transformation Stage'},
    width=900,
    height=600,
    template='plotly_dark'
)

fig_na_to_k_journey.show()

### Visualizing Pipeline Effects: 3D Before & After Feature Space

Now, let's visualize how the feature space changes for a selection of features (original vs. transformed) in 3D. We'll use 'Na_to_K', 'Age', and 'Sex' as representative features.

In [25]:
# Prepare original data for 3D plot
# Convert 'Sex' to numerical for 3D visualization (e.g., Female=0, Male=1)
X_original_3d = X[['Na_to_K', 'Age']].copy()
X_original_3d['Sex_numerical'] = X['Sex'].map({'F': 0, 'M': 1})
X_original_3d['Drug'] = y

# Need to apply the standard scaler for Age separately to get its transformed values
# The preprocessor.fit_transform(X) already did this, so we can extract directly from X_transformed_df
age_standard_scaler = preprocessor.named_transformers_['other_num'].named_steps['standard_scaler']
age_original_values_df = X[['Age']]
age_standard_scaled_values = age_standard_scaler.transform(age_original_values_df)

# Prepare transformed data for 3D plot
X_transformed_selected_df = pd.DataFrame({
    'Transformed_Na_to_K': na_to_k_standard_scaled_values.flatten(),
    'Transformed_Age': age_standard_scaled_values.flatten(),
    'Transformed_Sex_F': X_transformed_df['cat_feature__Sex_F'], # Directly use the OHE column
    'Drug': y
})

# --- 3D Plot: Original Feature Space ---
fig_original_3d = px.scatter_3d(
    X_original_3d,
    x='Na_to_K',
    y='Age',
    z='Sex_numerical',
    color='Drug',
    title='3D Original Feature Space (Na_to_K, Age, Sex)',
    labels={
        'Na_to_K': 'Original Na_to_K',
        'Age': 'Original Age',
        'Sex_numerical': 'Original Sex (0=F, 1=M)'
    },
    width=900,
    height=700,
    template='plotly_dark'
)
fig_original_3d.show()

# --- 3D Plot: Transformed Feature Space ---
fig_transformed_3d = px.scatter_3d(
    X_transformed_selected_df,
    x='Transformed_Na_to_K',
    y='Transformed_Age',
    z='Transformed_Sex_F',
    color='Drug',
    title='3D Transformed Feature Space (Na_to_K, Age, Sex_F)',
    labels={
        'Transformed_Na_to_K': 'Transformed Na_to_K',
        'Transformed_Age': 'Transformed Age',
        'Transformed_Sex_F': 'Transformed Sex (Female=1, Male=0)'
    },
    width=900,
    height=700,
    template='plotly_dark'
)
fig_transformed_3d.show()

In [26]:
print('Checking for missing values in the DataFrame:')
display(df.isnull().sum())

Checking for missing values in the DataFrame:


,0
Age,0
Sex,0
BP,0
Cholesterol,0
Na_to_K,0
Drug,0
